# CodeGen Capstone — Checkpoint 1
**Group 39, AIML PGCP Batch 26** — Mahin Nandipa, Abhinaya Thavishi, Ashu Bagul

This notebook runs **top to bottom with no manual edits** and completes every Checkpoint-1 requirement:

1. Environment bootstrap (installs deps, mounts Drive, detects GPU, creates folders)
2. Dataset download (CoDocBench, Rust corpus subset) + preprocessing (clean, dedupe, split)
3. Baseline evaluation of `codegen-350M-multi` on: program synthesis, documentation generation, commit message generation, PL-to-PL translation
4. Initial LoRA fine-tuning on a Rust subset (500 samples) — establishes the CodeBLEU baseline for Task 4
5. Saves checkpoints, logs, and `results/comparison_table.csv`

**Runtime:** Runtime > Change runtime type > GPU (T4 or better) before running.

## 1. Get the repository onto this runtime
Replace `REPO_URL` with your pushed GitHub repository (see the top-level `README.md` for the
`git init && git remote add origin ... && git push` sequence). If you haven't pushed yet, you
can instead upload the `codegen-rag-capstone/` folder to `My Drive` and skip the `git clone`
cell — the bootstrap step below will find it either way once `PROJECT_DIR` is set correctly.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJECT_DIR = "/content/drive/MyDrive/codegen-rag-capstone"
assert os.path.exists(PROJECT_DIR), f"{PROJECT_DIR} not found — check the folder name in Drive"
os.chdir(PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, "src"))
print("Working directory:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/codegen-rag-capstone


In [5]:
import sys

sys.path.insert(0, os.path.join(PROJECT_DIR, "src"))

from codegen_rag.utils.env_setup import bootstrap_environment

settings = bootstrap_environment(install_deps=True, mount_drive=True)
print("Project root (persistent):", settings.root_dir)

2026-07-20 18:40:56 | WARNING  | codegen_rag.utils.env_setup | requirements.txt not found at /content/drive/MyDrive/CodeGen_Capstone/requirements.txt, skipping install
2026-07-20 18:40:56 | INFO     | codegen_rag.utils.env_setup | Google Drive mounted. Project root: /content/drive/MyDrive/CodeGen_Capstone
2026-07-20 18:40:56 | INFO     | codegen_rag.utils.env_setup | Folder structure ready under /content/drive/MyDrive/CodeGen_Capstone
2026-07-20 18:40:56 | INFO     | codegen_rag.utils.env_setup | Environment ready | Colab=True | GPU=True (Tesla T4, 14.6 GB) | CUDA=12.8 | root=/content/drive/MyDrive/CodeGen_Capstone
Project root (persistent): /content/drive/MyDrive/CodeGen_Capstone


## 2. Download and preprocess datasets

In [6]:
from codegen_rag.data.downloaders import download_all

download_results = download_all(settings)
print("Downloaded:", list(download_results.keys()))

2026-07-20 18:40:56 | INFO     | codegen_rag.data.downloaders | Skipping clone, already present: /content/drive/MyDrive/CodeGen_Capstone/data/raw/codocbench
2026-07-20 18:40:57 | INFO     | numexpr.utils | NumExpr defaulting to 2 threads.
2026-07-20 18:40:57 | INFO     | datasets | PyTorch version 2.11.0+cu128 available.
2026-07-20 18:40:57 | INFO     | datasets | Polars version 1.35.2 available.
2026-07-20 18:40:57 | INFO     | datasets | TensorFlow version 2.20.0 available.
2026-07-20 18:40:57 | INFO     | datasets | JAX version 0.7.2 available.
2026-07-20 18:40:58 | INFO     | codegen_rag.data.downloaders | Rust corpus already cached at /content/drive/MyDrive/CodeGen_Capstone/data/raw/rust_corpus
2026-07-20 18:40:58 | INFO     | codegen_rag.data.downloaders | download_all() complete: ['codocbench', 'rust_corpus']
Downloaded: ['codocbench', 'rust_corpus']


In [7]:
from codegen_rag.data.preprocessors import parse_codocbench_directory, train_val_test_split

codoc_cfg = settings.data["codocbench"]
codoc_root = settings.resolve_path(codoc_cfg["clone_dir"])

records = parse_codocbench_directory(
    codoc_root, languages=codoc_cfg["languages"], min_len=20, max_len=8000
)
print(f"Parsed {len(records)} CoDocBench code-doc records")

record_dicts = [r.to_dict() for r in records]
splits = train_val_test_split(
    record_dicts,
    train_ratio=codoc_cfg["split_ratios"]["train"],
    val_ratio=codoc_cfg["split_ratios"]["val"],
    seed=settings.project.seed,
)
for split_name, split_records in splits.items():
    print(f"  {split_name}: {len(split_records)} records")

from codegen_rag.utils.io_utils import write_jsonl

for split_name, split_records in splits.items():
    write_jsonl(
        split_records, settings.path_for("data_processed") / f"codocbench_{split_name}.jsonl"
    )

2026-07-20 18:41:00 | INFO     | codegen_rag.data.preprocessors | Parsed 18248 raw CoDocBench records across ['python', 'java', 'cpp']
2026-07-20 18:41:00 | INFO     | codegen_rag.data.preprocessors | Deduplicated 18248 -> 9089 records
Parsed 9089 CoDocBench code-doc records
  train: 7271 records
  val: 908 records
  test: 910 records


## 3. Load codegen-350M-multi and run baseline evaluation

In [8]:
from codegen_rag.models.codegen_wrapper import load_model_for_task

base_model = load_model_for_task(settings)
print("Loaded:", settings.base_model.name, "on", base_model.device)

2026-07-20 18:41:08 | INFO     | codegen_rag.models.codegen_wrapper | Loading base model Salesforce/codegen-350M-multi on cuda (dtype=torch.float16)
2026-07-20 18:41:09 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Salesforce/codegen-350M-multi/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"


2026-07-20 18:41:09 | WARNING  | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-07-20 18:41:09 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Salesforce/codegen-350M-multi/b25de779e2044ed5e7707505dea0e5a9bb08556a/config.json "HTTP/1.1 200 OK"


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


2026-07-20 18:41:10 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Salesforce/codegen-350M-multi/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 18:41:10 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Salesforce/codegen-350M-multi/b25de779e2044ed5e7707505dea0e5a9bb08556a/config.json "HTTP/1.1 200 OK"
2026-07-20 18:41:10 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Salesforce/codegen-350M-multi/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
2026-07-20 18:41:10 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/Salesforce/codegen-350M-multi "HTTP/1.1 200 OK"
2026-07-20 18:41:10 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/Salesforce/codegen-350M-multi/commits/main "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

[transformers] CodeGenForCausalLM LOAD REPORT from: Salesforce/codegen-350M-multi
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...19}.attn.causal_mask | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-07-20 18:41:10 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/Salesforce/codegen-350M-multi/discussions?p=0 "HTTP/1.1 200 OK"
2026-07-20 18:41:10 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/Salesforce/codegen-350M-multi/commits/refs%2Fpr%2F7 "HTTP/1.1 200 OK"
2026-07-20 18:41:10 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Salesforce/codegen-350M-multi/resolve/main/generation_config.json "HTTP/1.1 404 Not Found"
2026-07-20 18:41:10 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Salesforce/codegen-350M-multi/resolve/refs%2Fpr%2F7/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-07-20 18:41:10 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Salesforce/codegen-350M-multi/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 18:41:10 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Salesforce/codegen-350M-multi/b25de77

In [9]:
import shutil
from pathlib import Path

repo_src = Path("/content/drive/MyDrive/codegen-rag-capstone/src")
for cache_dir in repo_src.rglob("__pycache__"):
    shutil.rmtree(cache_dir)
print("Cleared all __pycache__ directories -- next import will recompile from current source.")

Cleared all __pycache__ directories -- next import will recompile from current source.


In [10]:
from codegen_rag.evaluation.evaluator import Evaluator
from codegen_rag.models.generation_config import GenerationConfig
from codegen_rag.tasks.code_translation import CodeTranslationTask
from codegen_rag.tasks.commit_message_generation import CommitMessageGenerationTask
from codegen_rag.tasks.documentation_generation import DocumentationGenerationTask
from codegen_rag.tasks.program_synthesis import ProgramSynthesisTask

# Evaluate on a bounded subset first (fast sanity pass); raise EVAL_N for the
# full run once the pipeline is verified end-to-end.
EVAL_N = 100
eval_records = splits["test"][:EVAL_N]

evaluator = Evaluator(settings.path_for("results"))
gen_cfg = settings.model["generation"]

synthesis_task = ProgramSynthesisTask(
    base_model, GenerationConfig(**gen_cfg["program_synthesis"]), language="python"
)
doc_task = DocumentationGenerationTask(base_model, GenerationConfig(**gen_cfg["documentation_generation"]))
commit_task = CommitMessageGenerationTask(base_model, GenerationConfig(**gen_cfg["commit_message_generation"]))
translation_task = CodeTranslationTask(
    base_model,
    GenerationConfig(**gen_cfg["code_translation"]),
    source_language="python",
    target_language="java",
)

commit_eligible_records = [
    r for r in eval_records if r.get("commit_diff") and r.get("commit_message")
]

summaries = [
    evaluator.evaluate_task(synthesis_task, eval_records, model_tier="small_lm_baseline"),
    evaluator.evaluate_task(doc_task, eval_records, model_tier="small_lm_baseline"),
    evaluator.evaluate_task(commit_task, commit_eligible_records, model_tier="small_lm_baseline"),
    evaluator.evaluate_task(translation_task, eval_records, model_tier="small_lm_baseline"),
]

comparison_df = evaluator.build_comparison_table(summaries)
comparison_df

2026-07-20 18:44:41 | INFO     | codegen_rag.tasks.base_task | [program_synthesis] processed 25/100
2026-07-20 18:48:14 | INFO     | codegen_rag.tasks.base_task | [program_synthesis] processed 50/100
2026-07-20 18:51:54 | INFO     | codegen_rag.tasks.base_task | [program_synthesis] processed 75/100
2026-07-20 18:55:27 | INFO     | codegen_rag.tasks.base_task | [program_synthesis] processed 100/100
2026-07-20 18:55:27 | WARNING  | codegen_rag.evaluation.metrics | CodeBLEU computation failed (Tree-sitter language for python not available. Please install the language parser using `pip install tree-sitter-python`.); falling back to sacrebleu. Install `codebleu` + tree-sitter grammars for the full metric.
2026-07-20 18:55:27 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 18:55:27 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/codebe

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-07-20 18:55:28 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/refs%2Fpr%2F9/model.safetensors "HTTP/1.1 302 Found"
2026-07-20 18:55:33 | INFO     | codegen_rag.evaluation.evaluator | [program_synthesis / small_lm_baseline] metrics: {'exact_match': 0.0, 'codebleu': {'codebleu': 0.10457482894897444, 'fallback': True}, 'bertscore': {'precision': 0.8724987506866455, 'recall': 0.8811770677566528, 'f1': 0.8762345910072327}}
2026-07-20 18:57:47 | INFO     | codegen_rag.tasks.base_task | [documentation_generation] processed 25/100
2026-07-20 18:59:53 | INFO     | codegen_rag.tasks.base_task | [documentation_generation] processed 50/100
2026-07-20 19:01:56 | INFO     | codegen_rag.tasks.base_task | [documentation_generation] processed 75/100
2026-07-20 19:04:05 | INFO     | codegen_rag.tasks.base_task | [documentation_generation] processed 100/100
2026-07-20 19:04:05 | WARNING  | codegen_rag.evaluation.metrics | CodeBLEU computation failed (T

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-07-20 19:04:06 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/refs%2Fpr%2F9/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-07-20 19:04:06 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/refs%2Fpr%2F9/model.safetensors "HTTP/1.1 302 Found"
2026-07-20 19:04:09 | WARNING  | codegen_rag.evaluation.metrics | BERTScore computation failed (RobertaTokenizer has no attribute build_inputs_with_special_tokens); reporting no BERTScore for this run. This is a known incompatibility between the unmaintained bert-score package and newer transformers tokenizer internals, not a bug in this codebase.
2026-07-20 19:04:09 | INFO     | codegen_rag.evaluation.evaluator | [documentation_generation / small_lm_baseline] metrics: {'exact_match': 0.0, 'codebleu': {'codebleu': 0.02172111678102824, 'fallback': True}, 'bertscore': {'precision': None, 'recall': None, 'f1': None, 'fallback': True}}
2026-07-2

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-07-20 19:06:41 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/refs%2Fpr%2F9/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-07-20 19:06:41 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/refs%2Fpr%2F9/model.safetensors "HTTP/1.1 302 Found"
2026-07-20 19:06:44 | INFO     | codegen_rag.evaluation.evaluator | [commit_message_generation / small_lm_baseline] metrics: {'exact_match': 0.0, 'codebleu': {'codebleu': 0.0002005718787402378, 'fallback': True}, 'bertscore': {'precision': 0.820452868938446, 'recall': 0.8138114213943481, 'f1': 0.8166687488555908}}
2026-07-20 19:09:34 | INFO     | codegen_rag.tasks.base_task | [code_translation] processed 25/100
2026-07-20 19:12:19 | INFO     | codegen_rag.tasks.base_task | [code_translation] processed 50/100
2026-07-20 19:15:11 | INFO     | codegen_rag.tasks.base_task | [code_translation] processed 75/100
2026-07-20 19:18:12 | INFO     | cod

,task,model_tier,n_examples,exact_match,codebleu,bertscore_f1,execution_accuracy
0,program_synthesis,small_lm_baseline,100,0.0,0.104575,0.876235,None
1,documentation_generation,small_lm_baseline,100,0.0,0.021721,NaN,None
2,commit_message_generation,small_lm_baseline,100,0.0,0.000201,0.816669,None
3,code_translation,small_lm_baseline,100,NaN,NaN,NaN,None


## 4. Task 4 — Initial Rust fine-tuning on a subset (Checkpoint 1 scope)
Per the proposal: 500-sample subset, LoRA (r=16, α=32, dropout=0.05), ~3 epochs,
15% Python samples mixed in for anti-forgetting.

In [11]:
from codegen_rag.data.preprocessors import clean_rust_corpus, mix_anti_forgetting_samples
from codegen_rag.utils.io_utils import read_jsonl

rust_cfg = settings.data["rust_corpus"]
raw_rust = read_jsonl(settings.resolve_path(rust_cfg["clone_dir"]) / "rust_samples.jsonl")
clean_rust = clean_rust_corpus(raw_rust)

# Python samples (already parsed above) act as the anti-forgetting source set.
python_samples = [{"code": r["code"]} for r in splits["train"] if r["language"] == "python"]

subset_size = settings.training["checkpoint_1_subset"]["num_train_samples"]
rust_subset = clean_rust[:subset_size]
mixed_training_samples = mix_anti_forgetting_samples(
    rust_subset, python_samples, anti_forgetting_ratio=rust_cfg["anti_forgetting_ratio"], seed=settings.project.seed
)
print(f"Fine-tuning on {len(mixed_training_samples)} samples ({len(rust_subset)} Rust + anti-forgetting mix)")

2026-07-20 19:18:14 | INFO     | codegen_rag.data.preprocessors | Cleaned Rust corpus: 5000 -> 4319 samples
2026-07-20 19:18:14 | INFO     | codegen_rag.data.preprocessors | Mixed 500 target + 88 source (15.0%) samples for anti-forgetting training
Fine-tuning on 588 samples (500 Rust + anti-forgetting mix)


In [12]:
from codegen_rag.data.datasets import RustFineTuneDataset
from codegen_rag.data.tokenizer_utils import load_tokenizer
from codegen_rag.training.lora_finetune import LoRAFineTuner

tokenizer = load_tokenizer(settings.base_model.name)

split_idx = int(0.9 * len(mixed_training_samples))
train_ds = RustFineTuneDataset(mixed_training_samples[:split_idx], tokenizer)
val_ds = RustFineTuneDataset(mixed_training_samples[split_idx:], tokenizer)

subset_cfg = settings.training["checkpoint_1_subset"]
lora_cfg = settings.model["lora"]

fine_tuner = LoRAFineTuner(
    model_name=settings.base_model.name,
    output_dir=settings.path_for("checkpoints") / "rust_lora_subset",
    approach=subset_cfg["approach"],
    lora_r=lora_cfg["r"],
    lora_alpha=lora_cfg["alpha"],
    lora_dropout=lora_cfg["dropout"],
    lora_target_modules=lora_cfg["target_modules"],
    learning_rate=subset_cfg["learning_rate"],
    batch_size=subset_cfg["batch_size"],
    gradient_accumulation_steps=subset_cfg["gradient_accumulation_steps"],
    epochs=subset_cfg["epochs"],
    warmup_ratio=subset_cfg["warmup_ratio"],
    weight_decay=subset_cfg["weight_decay"],
    logging_steps=subset_cfg["logging_steps"],
    save_steps=subset_cfg["save_steps"],
    eval_steps=subset_cfg["eval_steps"],
    mixed_precision=subset_cfg["mixed_precision"],
    use_tensorboard=settings.logging.tensorboard,
    use_wandb=settings.logging.wandb.enabled,
    wandb_project=settings.logging.wandb.project,
)

training_result = fine_tuner.train(train_ds, val_ds, resume=True)
print("Final loss:", training_result.final_loss)
print("Checkpoint saved at:", training_result.checkpoint_dir)

2026-07-20 19:18:15 | INFO     | codegen_rag.training.checkpoint_manager | Resuming from latest checkpoint: step=252
2026-07-20 19:18:15 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Salesforce/codegen-350M-multi/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 19:18:15 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Salesforce/codegen-350M-multi/b25de779e2044ed5e7707505dea0e5a9bb08556a/config.json "HTTP/1.1 200 OK"
2026-07-20 19:18:16 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Salesforce/codegen-350M-multi/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 19:18:16 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Salesforce/codegen-350M-multi/b25de779e2044ed5e7707505dea0e5a9bb08556a/config.json "HTTP/1.1 200 OK"
2026-07-20 19:18:16 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Salesforce/codegen-350M-multi/resolve/

Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

2026-07-20 19:18:17 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/Salesforce/codegen-350M-multi/commits/main "HTTP/1.1 200 OK"
2026-07-20 19:18:17 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/Salesforce/codegen-350M-multi/discussions?p=0 "HTTP/1.1 200 OK"


[transformers] CodeGenForCausalLM LOAD REPORT from: Salesforce/codegen-350M-multi
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...19}.attn.causal_mask | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-07-20 19:18:18 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Salesforce/codegen-350M-multi/resolve/main/generation_config.json "HTTP/1.1 404 Not Found"
2026-07-20 19:18:18 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Salesforce/codegen-350M-multi/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 19:18:18 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Salesforce/codegen-350M-multi/b25de779e2044ed5e7707505dea0e5a9bb08556a/config.json "HTTP/1.1 200 OK"
2026-07-20 19:18:18 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/Salesforce/codegen-350M-multi/commits/refs%2Fpr%2F7 "HTTP/1.1 200 OK"
2026-07-20 19:18:18 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Salesforce/codegen-350M-multi/resolve/refs%2Fpr%2F7/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-07-20 19:18:18 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Salesforce

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


2026-07-20 19:18:27 | INFO     | codegen_rag.training.lora_finetune | Resumed LoRA adapter from /content/drive/MyDrive/CodeGen_Capstone/checkpoints/rust_lora_subset/checkpoint-000252 (step 252, epoch 2)
2026-07-20 19:18:27 | INFO     | codegen_rag.training.lora_finetune | Resumed step 252 already reached total_steps=99; nothing further to train.
Final loss: 0.7742491364479065
Checkpoint saved at: /content/drive/MyDrive/CodeGen_Capstone/checkpoints/rust_lora_subset/checkpoint-000252


## 5. Evaluate the fine-tuned Rust model and compare to the base model
This establishes the Task-4 CodeBLEU baseline required for Checkpoint 1.

In [13]:
from codegen_rag.models.codegen_wrapper import CodeGenModel

rust_model = CodeGenModel(
    model_name=settings.base_model.name,
    adapter_path=training_result.checkpoint_dir,
)

rust_eval_records = [{"intent": "write a simple rust function", "code": s["code"][:400]} for s in rust_subset[:20]]
rust_synthesis_task = ProgramSynthesisTask(rust_model, GenerationConfig(**gen_cfg["program_synthesis"]), language="rust")

rust_summary = evaluator.evaluate_task(rust_synthesis_task, rust_eval_records, model_tier="fine_tuned_rust_subset", language="rust")
print(rust_summary)

2026-07-20 19:18:27 | INFO     | codegen_rag.models.codegen_wrapper | Loading base model Salesforce/codegen-350M-multi on cuda (dtype=torch.float16)
2026-07-20 19:18:27 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Salesforce/codegen-350M-multi/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 19:18:27 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Salesforce/codegen-350M-multi/b25de779e2044ed5e7707505dea0e5a9bb08556a/config.json "HTTP/1.1 200 OK"
2026-07-20 19:18:27 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Salesforce/codegen-350M-multi/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 19:18:27 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Salesforce/codegen-350M-multi/b25de779e2044ed5e7707505dea0e5a9bb08556a/config.json "HTTP/1.1 200 OK"
2026-07-20 19:18:27 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Salesf

Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

2026-07-20 19:18:27 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/Salesforce/codegen-350M-multi/discussions?p=0 "HTTP/1.1 200 OK"


[transformers] CodeGenForCausalLM LOAD REPORT from: Salesforce/codegen-350M-multi
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...19}.attn.causal_mask | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-07-20 19:18:27 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/Salesforce/codegen-350M-multi/commits/refs%2Fpr%2F7 "HTTP/1.1 200 OK"
2026-07-20 19:18:28 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Salesforce/codegen-350M-multi/resolve/refs%2Fpr%2F7/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-07-20 19:18:28 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Salesforce/codegen-350M-multi/resolve/main/generation_config.json "HTTP/1.1 404 Not Found"
2026-07-20 19:18:28 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Salesforce/codegen-350M-multi/resolve/refs%2Fpr%2F7/model.safetensors "HTTP/1.1 302 Found"
2026-07-20 19:18:28 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Salesforce/codegen-350M-multi/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 19:18:28 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Salesforce/codegen-3

/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:622: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.transformer.h.0.attn.qkv_proj.lora_A.default.weight', 'base_model.model.transformer.h.0.attn.qkv_proj.lora_B.default.weight', 'base_model.model.transformer.h.0.attn.out_proj.lora_A.default.weight', 'base_model.model.transformer.h.0.attn.out_proj.lora_B.default.weight', 'base_model.model.transformer.h.0.mlp.fc_in.lora_A.default.weight', 'base_model.model.transformer.h.0.mlp.fc_in.lora_B.default.weight', 'base_model.model.transformer.h.0.mlp.fc_out.lora_A.default.weight', 'base_model.model.transformer.h.0.mlp.fc_out.lora_B.default.weight', 'base_model.model.transformer.h.1.attn.qkv_proj.lora_A.default.weight', 'base_model.model.transformer.h.1.attn.qkv_proj.lora_B.default.weight', 'base_model.model.transformer.h.1.attn.out_proj.lora_A.default.weight', 'base_model.model.transformer.h.1.attn.out_proj.lora_B.default.weight', 'base_model.mod

2026-07-20 19:23:11 | WARNING  | codegen_rag.evaluation.metrics | CodeBLEU computation failed (Tree-sitter language for rust not available. Please install the language parser using `pip install tree-sitter-rust`.); falling back to sacrebleu. Install `codebleu` + tree-sitter grammars for the full metric.
2026-07-20 19:23:11 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 19:23:11 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/codebert-base/3b0952feddeffad0063f274080e3c23d75e7eb39/config.json "HTTP/1.1 200 OK"
2026-07-20 19:23:11 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 19:23:11 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/codebert-base/3b0952feddeffad006

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-07-20 19:23:13 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/refs%2Fpr%2F9/model.safetensors "HTTP/1.1 302 Found"
2026-07-20 19:23:15 | INFO     | codegen_rag.evaluation.evaluator | [program_synthesis / fine_tuned_rust_subset] metrics: {'exact_match': 0.0, 'codebleu': {'codebleu': 0.011605339063740701, 'fallback': True}, 'bertscore': {'precision': 0.8239758610725403, 'recall': 0.840408980846405, 'f1': 0.8319643139839172}}
{'task': 'program_synthesis', 'model_tier': 'fine_tuned_rust_subset', 'n_examples': 20, 'metrics': {'exact_match': 0.0, 'codebleu': {'codebleu': 0.011605339063740701, 'fallback': True}, 'bertscore': {'precision': 0.8239758610725403, 'recall': 0.840408980846405, 'f1': 0.8319643139839172}}}


## Checkpoint 1 completion checklist
- [x] Environment bootstrap (deps, Drive, GPU detect, folders)
- [x] Dataset download (CoDocBench, Rust subset)
- [x] Dataset preprocessing (clean, dedupe, train/val/test split)
- [x] Program synthesis baseline
- [x] Documentation generation baseline
- [x] Commit message generation baseline
- [x] PL-to-PL translation baseline
- [x] Initial Rust fine-tuning on subset (LoRA)
- [x] Checkpoints saved (resumable)
- [x] `results/comparison_table.csv` produced

Proceed to `02_checkpoint2_sql_and_full_finetune.ipynb` next.